## Декораторы

In [19]:
import sys
import typing as tp


def deprecate(func):
    def wrapper(*args, **kwargs):
        print(
            f"{func.__name__} is deprecated",
            file=sys.stderr
        )
        return func(*args, **kwargs)
    return wrapper


pprint = deprecate(print)

pprint([1, 2, 3])

[1, 2, 3]


print is deprecated


In [20]:
@deprecate
def pprint(*args: tp.Any, **kwargs: tp.Any):
    '''
    Старый pprint
    '''
    print(*args, **kwargs)

In [21]:
pprint([1, 2, 3])

[1, 2, 3]


pprint is deprecated


In [22]:
help(pprint)

Help on function wrapper in module __main__:

wrapper(*args, **kwargs)



In [5]:
pprint.__name__, pprint.__doc__, pprint.__module__

('wrapper', None, '__main__')

### Решение 1

In [6]:
import sys


def deprecate(func):
    def wrapper(*args, **kwargs):
        print(
            f"{func.__name__} is deprecated",
            file=sys.stderr
        )
        return func(*args, **kwargs)
    wrapper.__name__ = func.__name__
    wrapper.__doc__ = func.__doc__
    wrapper.__module__ = func.__module__
    return wrapper


pprint = deprecate(print)

pprint([1, 2, 3])

[1, 2, 3]


print is deprecated


In [7]:
@deprecate
def pprint(*args: tp.Any, **kwargs: tp.Any):
    '''
    Старый pprint
    '''
    print(*args, **kwargs)

In [8]:
help(pprint)

Help on function pprint in module __main__:

pprint(*args, **kwargs)
    Старый pprint



In [9]:
pprint.__name__, pprint.__doc__, pprint.__module__

('pprint', '\n    Старый pprint\n    ', '__main__')

### Решение 2

In [10]:
import functools

import sys


def deprecate(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print('{} is deprecated'.format(func.__name__),
              file=sys.stderr)
        return func(*args, **kwargs)
    return wrapper


pprint = deprecate(print)

pprint([1, 2, 3])

[1, 2, 3]


print is deprecated


In [11]:
@deprecate
def pprint(*args: tp.Any, **kwargs: tp.Any):
    '''
    Старый pprint
    '''
    print(*args, **kwargs)

In [12]:
help(pprint)

Help on function pprint in module __main__:

pprint(*args: Any, **kwargs: Any)
    Старый pprint



In [13]:
pprint.__name__, pprint.__doc__, pprint.__module__

('pprint', '\n    Старый pprint\n    ', '__main__')

## LEGB

- Local
- Enclosing
- Global
- Built-in

![](./images/legb.png)

## Closures

In computer programming languages, a closure is a function together with a referencing environment of that function. A closure function is any function that uses a variable that is defined in an environment (or scope) that is external to that function, and is accessible within the function when invoked from a scope in which that free variable is not defined.

In [30]:
def make_adder(x):
    def adder(y):
        return x + y
    return adder

In [ ]:
add_two  = make_adder(2)
add_five = make_adder(5)

add_two(7) + add_five(10)

## Декоратор принимающий аргументы

In [14]:
import time


def count_time(f, iters=10):
    def wrapper(*args, **kwargs):
        timings = []
        for _ in range(iters):
            start = time.time()
            res = f(*args, **kwargs)
            end = time.time()
            timings.append(end - start)
        print(f"Function {f.__name__} took {sum(timings) / iters} seconds")
        return res

    return wrapper


@count_time(iters=5)
def fibonacci_new(n=30):
    a = [1 for i_ in range(n)]
    for i in range(2, n):
        a[i] = a[i - 1] + a[i - 2]
    return a[-1]


fibonacci_new(10)

TypeError: count_time() missing 1 required positional argument: 'f'

In [15]:
import time


def batch_count(iters=10):
    def count_time(f):
        def wrapper(*args, **kwargs):
            timings = []
            for _ in range(iters):
                start = time.time()
                res = f(*args, **kwargs)
                end = time.time()
                timings.append(end - start)
            print(f"Function {f.__name__} took {sum(timings) / iters} seconds in {iters} iteations")
            return res

        return wrapper
    return count_time


@batch_count(iters=10)
def fibonacci(n=30):
    a = [1 for i_ in range(n)]
    for i in range(2, n):
        a[i] = a[i - 1] + a[i - 2]
    return a[-1]

ffibonacci = batch_count(iters=10)(fibonacci)


fibonacci(10)

Function fibonacci took 2.789497375488281e-06 seconds in 10 iteations


55

In [17]:
import functools
import sys


def trace(dest=sys.stderr):
    def wraps(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            print(
                f'{func.__name__} called with args {args}, {kwargs}!',
                file=dest
            )
            return func(*args, **kwargs)
        return wrapper
    return wraps


@trace()
def f(x, test):
    if test > 1:
        return f(x, test / 2)


f('Hi!', test=42)

f called with args ('Hi!',), {'test': 42}!
f called with args ('Hi!', 21.0), {}!
f called with args ('Hi!', 10.5), {}!
f called with args ('Hi!', 5.25), {}!
f called with args ('Hi!', 2.625), {}!
f called with args ('Hi!', 1.3125), {}!
f called with args ('Hi!', 0.65625), {}!


## Аттрибуты объекта функции

In [18]:
import functools


def count_calls(func):
    func.calls = 1
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(
            f'{func.__name__} called {func.calls} time(s) with args {args}, {kwargs}!',
        )
        func.calls += 1
        return func(*args, **kwargs)
    return wrapper


@count_calls
def f(x, test):
    if test > 1:
        return f(x, test / 2)


f('Hi!', test=42)

f called 1 time(s) with args ('Hi!',), {'test': 42}!
f called 2 time(s) with args ('Hi!', 21.0), {}!
f called 3 time(s) with args ('Hi!', 10.5), {}!
f called 4 time(s) with args ('Hi!', 5.25), {}!
f called 5 time(s) with args ('Hi!', 2.625), {}!
f called 6 time(s) with args ('Hi!', 1.3125), {}!
f called 7 time(s) with args ('Hi!', 0.65625), {}!
